# Day 3 · 실습 03. DINOv3 Image Embedding과 유사 이미지 검색

**연계 이론:** 이론 3(Self-Supervised Learning과 DINOv3)

사전학습된 DINOv3의 CLS Token에서 Image Embedding을 추출합니다.
Cosine Similarity를 계산하고 Heatmap과 Top-k 검색 결과를 관찰합니다.
DINOv3 모델은 새로 학습하지 않습니다.

**사전 준비:** DINOv3 공식 모델은 접근 승인이 필요할 수 있습니다.
모델 페이지: https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m
승인 후 터미널에서 `hf auth login`으로 개인 계정에 로그인하세요.
필요 패키지: torch, torchvision, transformers>=4.56.0, huggingface_hub,
pillow, numpy, pandas, matplotlib. Notebook에 토큰을 직접 입력하지 않습니다.

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoImageProcessor, AutoModel

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path(os.getenv("DAY3_PROJECT_ROOT", str(Path.cwd()))).expanduser().resolve()
ASSETS_DIR = PROJECT_ROOT / "assets"
WEIGHTS_DIR = PROJECT_ROOT / "weights"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_ROOT = Path(os.getenv("DAY3_DATA_ROOT", str(ASSETS_DIR / "hymenoptera_data"))).expanduser().resolve()
MODEL_ID = os.getenv("DAY3_DINOV3_MODEL", "facebook/dinov3-vits16-pretrain-lvd1689m")
for folder in (ASSETS_DIR, WEIGHTS_DIR, OUTPUTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| DINOv3:", MODEL_ID)
print("데이터:", DATA_ROOT)

## 1. 개미·벌 데이터셋 준비

실습 02와 같은 경로의 데이터를 재사용합니다.
기존 데이터가 없을 경우 PyTorch 공식 튜토리얼 데이터를 다운로드합니다.

In [ ]:
DATA_URL = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"

def ensure_dataset(root=DATA_ROOT):
    required = [root / split / name for split in ("train", "val") for name in ("ants", "bees")]
    if all(p.is_dir() for p in required):
        print("기존 데이터 사용:", root)
        return root

    root.parent.mkdir(parents=True, exist_ok=True)
    archive = root.parent / "hymenoptera_data.zip"
    if not archive.exists():
        print("개미·벌 데이터 다운로드 중...")
        temp = archive.with_suffix(".zip.part")
        try:
            with urllib.request.urlopen(DATA_URL, timeout=90) as response, temp.open("wb") as out:
                shutil.copyfileobj(response, out)
            temp.replace(archive)
        finally:
            temp.unlink(missing_ok=True)

    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            parts = Path(member.filename).parts
            if not parts or parts[0] != "hymenoptera_data":
                raise ValueError("예상하지 못한 ZIP 경로: " + member.filename)
            if len(parts) == 1:
                continue
            destination = (root / Path(*parts[1:])).resolve()
            if not destination.is_relative_to(root.resolve()):
                raise ValueError("허용되지 않은 ZIP 경로: " + member.filename)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst)

    if not all(p.is_dir() for p in required):
        raise FileNotFoundError(f"train/val/ants/bees 폴더를 확인하세요: {root}")
    print("데이터 준비 완료:", root)
    return root

ensure_dataset()

## 2. 이미지 12장 선택

검증 이미지에서 개미·벌 각각 6장을 고정된 Random Seed로 선택합니다.
정답 레이블은 출력 해석에만 사용하며 DINOv3에는 전달하지 않습니다.

In [ ]:
IMAGES_PER_CLASS = 6
image_paths, true_labels = [], []
for class_name in ("ants", "bees"):
    candidates = sorted((DATA_ROOT / "val" / class_name).glob("*.jpg"))
    if len(candidates) < IMAGES_PER_CLASS:
        raise ValueError(f"{class_name}: 이미지가 부족합니다.")
    chosen = rng.choice(len(candidates), size=IMAGES_PER_CLASS, replace=False)
    for idx in chosen:
        image_paths.append(candidates[int(idx)])
        true_labels.append(class_name)

images = []
for path in image_paths:
    with Image.open(path) as im:
        images.append(im.convert("RGB"))
short_labels = [f"{name}-{i+1}" for name in ("ants", "bees")
                for i in range(IMAGES_PER_CLASS)]

fig, axes = plt.subplots(2, IMAGES_PER_CLASS, figsize=(15, 5))
for ax, im, label in zip(axes.flat, images, short_labels):
    ax.imshow(im)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("선택 이미지:", len(images))

## 3. DINOv3 모델 불러오기

`AutoImageProcessor`는 전처리, `AutoModel`은 사전학습 Backbone을 불러옵니다.
`DAY3_DINOV3_MODEL`에 승인된 로컬 모델 폴더를 지정할 수도 있습니다.

In [ ]:
try:
    processor = AutoImageProcessor.from_pretrained(
        MODEL_ID, cache_dir=str(WEIGHTS_DIR / "huggingface")
    )
    model = AutoModel.from_pretrained(
        MODEL_ID, cache_dir=str(WEIGHTS_DIR / "huggingface")
    ).to(DEVICE).eval()
except Exception as exc:
    raise RuntimeError(
        "DINOv3 모델 로딩 실패: 접근 승인, hf auth login, 인터넷 연결, "
        "모델 경로와 transformers 버전을 확인하세요."
    ) from exc
print("Model:", model.__class__.__name__)
print("Patch size:", model.config.patch_size)
print("Hidden size:", model.config.hidden_size)
print("Register tokens:", model.config.num_register_tokens)

## 4. Token Shape와 Global Image Embedding

DINOv3의 출력은 CLS Token, Register Token, Patch Token을 포함합니다.
이미지 전체의 유사도 비교에는 CLS Token 하나를 사용합니다.

In [ ]:
INFERENCE_BATCH_SIZE = 4
all_cls_vectors = []
for start in range(0, len(images), INFERENCE_BATCH_SIZE):
    batch = images[start:start + INFERENCE_BATCH_SIZE]
    inputs = processor(images=batch, return_tensors="pt")
    inputs = {name: value.to(DEVICE) for name, value in inputs.items()}
    with torch.inference_mode():
        hidden = model(**inputs).last_hidden_state

    if start == 0:
        _, _, height, width = inputs["pixel_values"].shape
        patch_size = model.config.patch_size
        patch_h, patch_w = patch_size if isinstance(patch_size, (tuple, list)) else (patch_size, patch_size)
        num_patch = (height // patch_h) * (width // patch_w)
        num_register = model.config.num_register_tokens
        expected_tokens = 1 + num_register + num_patch

        cls_tokens = hidden[:, :1, :]
        register_tokens = hidden[:, 1:1 + num_register, :]
        patch_tokens = hidden[:, 1 + num_register:, :]
        assert hidden.shape[1] == expected_tokens
        assert patch_tokens.shape[1] == num_patch

        print("입력 Shape:", tuple(inputs["pixel_values"].shape))
        print("전체 Token:", tuple(hidden.shape))
        print("CLS Token:", tuple(cls_tokens.shape))
        print("Register Tokens:", tuple(register_tokens.shape))
        print("Patch Tokens:", tuple(patch_tokens.shape))

    all_cls_vectors.append(hidden[:, 0, :].detach().float().cpu())

embeddings = torch.cat(all_cls_vectors, dim=0)
normalized = F.normalize(embeddings, p=2, dim=1)
print("Global Embeddings:", tuple(embeddings.shape))
print("정규화 후 첫 벡터 크기:", round(normalized[0].norm().item(), 6))

## 5. Cosine Similarity와 Heatmap

L2 정규화한 벡터의 내적은 Cosine Similarity와 같습니다.
행렬 대각선은 같은 이미지를 비교한 결과이므로 약 1입니다.

In [ ]:
similarity = (normalized @ normalized.T).numpy()
assert similarity.shape == (len(images), len(images))
assert np.allclose(np.diag(similarity), 1.0, atol=1e-5)
similarity_df = pd.DataFrame(similarity, index=short_labels, columns=short_labels)
print(similarity_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(8.5, 7))
heatmap = ax.imshow(similarity, vmin=-1, vmax=1, cmap="viridis")
ax.set_xticks(range(len(short_labels)), short_labels, rotation=65, ha="right")
ax.set_yticks(range(len(short_labels)), short_labels)
ax.set_title("DINOv3 Image-Image Cosine Similarity")
fig.colorbar(heatmap, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

## 6. Top-k 유사 이미지 검색

`QUERY_INDEX`는 기준 이미지, `TOP_K`는 검색할 이미지 수입니다.
기준 이미지 자기 자신은 검색 결과에서 제외합니다.

In [ ]:
QUERY_INDEX = 0  # 0 ~ 11 사이에서 변경 가능
TOP_K = 3
if not 0 <= QUERY_INDEX < len(images):
    raise ValueError("QUERY_INDEX 범위를 확인하세요.")
if not 1 <= TOP_K < len(images):
    raise ValueError("TOP_K는 1 이상, 전체 이미지 수 미만이어야 합니다.")

query_scores = similarity[QUERY_INDEX].copy()
query_scores[QUERY_INDEX] = -np.inf
ranked = np.argsort(query_scores)[::-1][:TOP_K]

fig, axes = plt.subplots(1, TOP_K + 1, figsize=(3.5 * (TOP_K + 1), 3.5))
axes[0].imshow(images[QUERY_INDEX])
axes[0].set_title(f"Query: {short_labels[QUERY_INDEX]}")
axes[0].axis("off")
for rank, (idx, ax) in enumerate(zip(ranked, axes[1:]), start=1):
    ax.imshow(images[idx])
    ax.set_title(f"Top {rank}: {short_labels[idx]}\ncos={similarity[QUERY_INDEX, idx]:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()
for rank, idx in enumerate(ranked, start=1):
    print(f"{rank}위: {short_labels[idx]} | cos={similarity[QUERY_INDEX, idx]:.4f} | "
          f"실제 클래스={true_labels[idx]}")

## 7. 결과 저장 및 요약

- Image Embedding과 유사도 행렬을 저장합니다.
- CLS Token에서 추출한 Embedding으로 이미지 유사도를 계산했습니다.
- 이번 실습에서는 DINOv3의 가중치를 학습하지 않았습니다.
- 다음 실습에서는 CLIP으로 이미지와 텍스트 사이의 유사도를 비교합니다.

In [ ]:
np.save(OUTPUTS_DIR / "day3_03_dinov3_cls_embeddings.npy", embeddings.numpy())
similarity_df.to_csv(OUTPUTS_DIR / "day3_03_dinov3_similarity.csv", encoding="utf-8-sig")
print("결과 저장:", OUTPUTS_DIR)